<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_pinn_steam_channel_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.2 — The Gas Channel of a Solid Oxide Cell

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L10.2 · Solid Oxide Cells and Optimisation**

A solid oxide cell at 800 °C splits steam into hydrogen. Steam enters a gas
channel, is used up along the electrode, and what is left comes out with the
hydrogen. How much of the steam is used at a given cell voltage, how much
hydrogen is made, and at what power? The current is not given: it is whatever
the voltage drives, and it falls along the channel as the steam runs short.
You solve the channel three ways - a boundary-value solver, finite volumes,
and a physics-informed network trained over a whole range of voltages - and
compare them on accuracy and on time. Then the network's own gradient finds
the voltage at which an hour of operation is worth most.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and finite volumes for an agreed 0.005 | — |
| **5** | a PINN for every voltage at once, and the comparison | the trial function, the residual |
| **6** | the best cell voltage, by a gradient through the network | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About half a minute on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex10.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

### The solid oxide electrolyser

A **solid oxide electrolysis cell (SOEC)** splits steam into hydrogen and
oxygen at 700 to 850 °C. Its electrolyte is a thin ceramic (typically
yttria-stabilised zirconia) that conducts oxide ions O²⁻ and nothing else.
Steam enters the fuel electrode, a porous nickel–ceramic layer, takes two
electrons from the power supply and leaves as hydrogen; the oxide ion crosses
the electrolyte and leaves the air electrode as oxygen. The same cell run
backwards is a solid oxide fuel cell (SOFC), the cell of Ex_09.2.

**Why the high temperature pays.** Splitting water takes a fixed energy, the
enthalpy $\Delta H$; at high temperature a larger part of it, $T\Delta S$, can be
supplied as heat and a smaller part, $\Delta G$, must be electricity. If the
steam is raised by heat from a neighbouring process, an SOEC needs about a
quarter less electricity per kilogram of hydrogen than a low-temperature
electrolyser. The electrode reactions are also fast when hot, so no platinum
or iridium is needed. The same stack can co-electrolyse steam and CO₂ to
syngas, or run in reverse as a fuel cell.

| | alkaline | PEM | solid oxide |
|---|---|---|---|
| electrolyte | KOH solution | polymer membrane | ceramic (zirconia) |
| ion through it | OH⁻ | H⁺ | O²⁻ |
| temperature | 60 – 90 °C | 50 – 80 °C | 700 – 850 °C |
| catalysts | nickel | platinum, iridium | nickel, ceramic oxides |
| electricity per kg H₂ | about 50 – 55 kWh | about 50 – 55 kWh | about 40 kWh, steam supplied |
| following a varying load | moderate | fast | slow; kept hot on standby |
| maturity | large plants | large plants | first plants of a few MW |

*Typical values from the literature, rounded; they vary with the stack and the
operating point.*

**The main challenges.**

- **Degradation.** Ceramic and metal layers age faster at 800 °C than in a
  low-temperature cell: nickel particles coarsen and move in the fuel
  electrode, the air electrode can delaminate at high current, and chromium
  from the steel interconnects poisons it. Stack lifetime is the main open
  question.
- **Heating and cycling.** A ceramic stack is heated slowly to avoid cracking
  and sealing failures, so it follows wind and solar poorly and is kept hot
  between uses.
- **Steam and heat.** The plant needs a steam supply and heat exchangers; the
  efficiency gain is real only where heat is available.
- **Gas composition along the cell.** With too much steam and too little
  hydrogen the nickel oxidises (hence the 10 % hydrogen in the feed below);
  where the steam runs short towards the outlet, the local voltage loss rises
  and the electrode ages faster. This is what the gas channel of this notebook
  decides.

The cell below draws the whole cell; the rest of the notebook is about its top
layer, the hydrogen (fuel) channel.

In [ ]:
pb.draw_soec(); plt.show()                                   # the whole cell: channel, electrodes, electrolyte, air side

**What you should see.** Steam goes down into the fuel electrode and hydrogen
comes back up into the channel, one for one; the oxide ions go down through
the electrolyte and leave as oxygen in the air channel; the power supply
drives the electrons into the fuel electrode.

### The hydrogen channel

A solid oxide cell with 100 × 100 mm of electrode runs as an electrolyser at
800 °C. A mixture of 90 % steam and 10 % hydrogen enters its gas channel at
1 m/s. The cell is held at a voltage $V$; along the channel the steam is split
and hydrogen takes its place. Wanted: the steam fraction along the channel,
the share of the steam that is used, the hydrogen made and the electrical
power - at the thermoneutral voltage, 1.29 V, and then at whichever voltage
pays best.

## 2 · The data

| | |
|---|---|
| cell | 800 °C, 100 mm long, 100 mm wide |
| gas channel | Ex_09.2's channel: 1 mm high, gas at 1 m/s, 90 % steam at the inlet, 1 atm |
| cell resistance | ASR = 0.25 Ω cm², all losses in one number — **an estimate** |
| gas diffusivity | 8.0e-04 m²/s, as in Ex_09.2 — **an estimate** |
| standard potential | $E_0$ = 0.99 V at 800 °C; thermoneutral voltage 1.29 V |
| cell voltage | 1.29 V in sections 4 and 5; 1.10 to 1.40 V for the network |

The thermodynamics is standard. The two estimates are of the right order and
were not measured on any cell; every result below depends on them.

## 3 · The physics

**The Nernst potential** is the voltage at which nothing happens; it depends
on the gas (L10.2's *Polarisation Curve*). With $y$ the steam fraction,

$$E(y) = E_0 + \frac{RT}{2F}\,\ln\frac{(1-y)\,\sqrt{p_{O_2}}}{y}$$

It **rises** as the steam is used up. **The local current** is what the cell
voltage drives against it, $i = (V - E(y))/\mathrm{ASR}$, so it falls along
the channel: the cell starves towards the outlet.

**The steam balance** along the channel is the convection-diffusion equation
of L9.2 and Ex_09.2 with a sink, the steam the current uses (L10.2's *Gas Transport Along
the Channel*):

$$u\,y_x = D\,y_{xx} - \frac{i(y)}{2F\,h\,c_{\mathrm{tot}}}$$

**Scaled units.** With $x$ over the channel length,

$$y_x = \frac{1}{\mathrm{Pe}}\,y_{xx} - K\,\big(V - E(y)\big), \qquad \mathrm{Pe} = \frac{uL}{D} = 125, \qquad K = \frac{L}{u\,h\,c_{\mathrm{tot}}\,2F\,\mathrm{ASR}} = 1.83\ \mathrm{V^{-1}}$$

with $y = 0.90$ at the inlet and $y_x = 0$ at the outlet. The equation is
nonlinear, through the logarithm in $E(y)$. The cell below prints the data,
draws the channel alone, along its length, and draws the Nernst potential
against the steam fraction.

In [ ]:
pb.describe_problem()
pb.draw_cell(); plt.show()                                   # the steam channel and the cell under it
yy = np.linspace(0.02, 0.98, 200)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(yy, pb.nernst(yy), lw=2)
ax.axhline(pb.V_TN, color="tab:orange", ls="--", lw=1, label="the cell voltage, 1.29 V")
ax.axvline(pb.Y_IN, color="k", ls=":", lw=1, label="the inlet, 90 % steam")
ax.set_xlabel("steam fraction  y"); ax.set_ylabel("Nernst potential  [V]"); ax.invert_xaxis()
ax.legend(frameon=False); plt.tight_layout(); plt.show()

**What you should see.** The data of section 2, a Nernst potential of 0.85 V
at the inlet, Pe = 125 and K = 1.83 per volt, and a steam supply that would
take 197 A to use completely; and a curve that climbs as the steam fraction
falls (read it from right to left): the gap to the cell voltage, which drives
the current, closes along the channel.

---

## 4 · The reference, and finite volumes

**The reference.** There is no exact solution, so the methods are scored
against a **converged numerical solution**: SciPy's boundary-value solver, a
collocation method that refines its own mesh until the residual is below a
tolerance. `pb.reference(V)` returns it as a function $y(x)$. The cell below
solves it at two tolerances and prints how far apart they are.

In [ ]:
V_CELL = pb.V_TN                                              # 1.29 V: the thermoneutral voltage
xs = np.linspace(0, 1, 1001)                                  # the points everything is compared on
t0 = time.perf_counter(); ref = pb.reference(V_CELL); t_ref = time.perf_counter() - t0
y_ref = ref(xs)
loose = pb.reference(V_CELL, tol=1e-6)(xs)
print(f"  tolerance 1e-06 against 1e-10: at most {np.abs(loose - y_ref).max():.1e} apart in steam fraction")
n_ref = pb.cell_numbers(xs, y_ref, V_CELL)
print(f"  at {V_CELL:.2f} V: steam at the outlet {n_ref['outlet']:.2f}, {100 * n_ref['used']:.0f} % of it used, "
      f"mean current {n_ref['i_mean']:.2f} A/cm2, hydrogen {n_ref['hydrogen']:.2f} g/h, power {n_ref['power']:.0f} W   ({t_ref:.2f} s)")

**What you should see.** The two tolerances agree to about 1e-12 in steam
fraction - **far better than the accuracy agreed below**, so this is the
reference. At 1.29 V the steam fraction falls from 0.90 to 0.27, 69 % of the
steam is used, the mean current density is 1.38 A/cm², and the cell makes
5.21 g of hydrogen an hour for 178 W.

**Finite volumes.** The channel is cut into cells. Each keeps count of its
steam: what the flow brings in from upstream, what diffusion exchanges with
both neighbours, and what the current uses. The convection takes the upstream
value (**upwind**), which is robust and first order. The equation is
nonlinear, so the cells are solved together by Newton's method.

**The agreed accuracy is 0.005** in steam fraction, half a per cent, anywhere
along the channel. The cell below finds the coarsest grid that meets it.

In [ ]:
def fvm_channel(n, V):
    # n cells; returns the cell centres and the steam fraction in each
    h = 1.0 / n; xc = (np.arange(n) + 0.5) * h
    d = 1.0 / (pb.PE * h * h)                                  # diffusion between neighbouring cells
    d_in = np.full(n, d); d_in[0] = 2 * d                      # the inlet face is half a cell away
    slope = pb.R_GAS * pb.T_CELL / (2 * pb.FARADAY)            # the size of the logarithm's derivative
    y = np.full(n, pb.Y_IN)
    for _ in range(50):                                        # Newton's method
        up = np.r_[pb.Y_IN, y[:-1]]                            # the upstream neighbour; the inlet for the first cell
        down = np.r_[y[1:], y[-1]]                             # the downstream neighbour; zero gradient at the outlet
        res = (y - up) / h - (d * (down - y) - d_in * (y - up)) + pb.K_REACT * (V - pb.nernst(y))
        yc = np.clip(y, 1e-4, 1 - 1e-4)
        dE = -slope * (1 / (1 - yc) + 1 / yc)                  # dE/dy
        main = 1 / h + d * np.r_[np.ones(n - 1), 0] + d_in - pb.K_REACT * dE
        J = sp.diags([-(1 / h + d_in[1:]), main, -d * np.ones(n - 1)], [-1, 0, 1]).tocsc()
        step = spla.spsolve(J, -res); y = y + step
        if np.abs(step).max() < 1e-12:
            break
    return xc, y

TARGET = 0.005                                                # the agreed accuracy, in steam fraction
for n in (25, 50, 100, 200):
    t0 = time.perf_counter(); xc, y_f = fvm_channel(n, V_CELL); sec = time.perf_counter() - t0
    err = np.abs(y_f - ref(xc)).max()
    print(f"  {n:3d} cells: steam in the last cell {y_f[-1]:.3f}, worst error {err:.4f}, {1000 * sec:.0f} ms"
          + ("   <- meets 0.005" if err <= TARGET else ""))
N_FVM = 50                                                    # the coarsest grid that meets it

**What you should see.** The error falling with every doubling of the
cells - 0.0106, 0.0042, 0.0019 and 0.0013 - which is first order, the price of
the upwind value. **50 cells are the first under 0.005**, in a millisecond or
two.

---

## 5 · A physics-informed network, and the comparison

**The network takes the cell voltage as an input.** It maps $(x, v)$ to the
steam fraction, where $v$ runs from 0 to 1 as the voltage runs from 1.10 to
1.40 V: one network for every voltage in that range, not one per voltage. It
is four layers of 32 tanh neurons, trained on 3000 collocation points spread
over the channel and the voltage range.

**The inlet is built in.** The trial function

$$\hat y(x, v) = y_{\mathrm{in}} + x\;\mathcal{N}(x, v)$$

is exactly 0.90 at the inlet, at every voltage. The outlet condition,
$\hat y_x = 0$, stays a loss term.

**TODO 1** is the trial function; **TODO 2** the residual of the scaled steam
balance, $\hat y_x - \hat y_{xx}/\mathrm{Pe} + K\,(V - E(\hat y))$.

In [ ]:
class Channel(torch.nn.Module):                               # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # dense tanh network: x and the scaled voltage in
    def forward(self, p):
        x = p[:, 0:1]
        return pb.Y_IN + x * self.net(p)                      # the trial function: the inlet value, exactly

In [ ]:
def residual(model, p):
    y = model(p)                                              # the network's steam fraction
    y_x = grad(y, p)[:, 0:1]                                  # its derivative along the channel, by autograd
    V = pb.voltage(p[:, 1:2])                                 # the cell voltage at each point, in volts
    return y_x - d2(y, p, 0) / pb.PE + pb.K_REACT * (V - pb.nernst(y))   # the steam balance; y_xx by autograd

In [ ]:
pts = to_tensor(pb.sample_channel(3000), requires_grad=True)  # collocation points (x, v), differentiable
out = to_tensor(pb.outlet_points(100), requires_grad=True)    # the outlet at 100 voltages, differentiable

model = Channel().to(DEVICE)                                  # four layers of 32
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio
def loss_fn():
    outlet = grad(model(out), out)[:, 0:1]                    # the gradient at the outlet, which should be zero
    return residual(model, pts).pow(2).mean() + outlet.pow(2).mean()   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, loss_fn, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s, for every voltage from {pb.V_LOW:.2f} to {pb.V_HIGH:.2f} V")

In [ ]:
def pinn_at(V, x=xs):
    # the network's steam fraction along the channel at cell voltage V
    v = (V - pb.V_LOW) / (pb.V_HIGH - pb.V_LOW)
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(to_tensor(np.c_[x, np.full(len(x), v)]))).ravel()   # y on the comparison points
t0 = time.perf_counter(); xc, y_f = fvm_channel(N_FVM, V_CELL); t_fvm = time.perf_counter() - t0   # finite volumes
y_pin = pinn_at(V_CELL)
t_pin = middle(lambda: pinn_at(V_CELL), reps=5)
e_fvm, e_pin = np.abs(y_f - ref(xc)).max(), np.abs(y_pin - y_ref).max()
n_fvm, n_pin = pb.cell_numbers(xc, y_f, V_CELL), pb.cell_numbers(xs, y_pin, V_CELL)

print(f"at {V_CELL:.2f} V{'':24s}{'steam out':>10s}{'used':>8s}{'hydrogen':>12s}{'power':>9s}{'worst error':>13s}{'one solution':>14s}{'training':>10s}")
for name, n, e, t, tr in (("reference, boundary-value solver", n_ref, None, t_ref, None),
                          ("finite volumes, 50 cells", n_fvm, e_fvm, t_fvm, None),
                          ("PINN, 4 x 32, any voltage", n_pin, e_pin, t_pin, train_time)):
    print(f"  {name:32s}{n['outlet']:10.2f}{100 * n['used']:6.0f} %{n['hydrogen']:8.2f} g/h{n['power']:7.0f} W"
          + (f"{e:13.4f}" if e is not None else f"{'-':>13s}") + f"{t:12.3f} s" + (f"{tr:8.0f} s" if tr else f"{'-':>10s}"))
print("the same network at other voltages, against the reference:")
for V in (1.10, 1.20, 1.35, 1.40):
    print(f"  {V:.2f} V: worst error {np.abs(pinn_at(V) - pb.reference(V)(xs)).max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for V, col in ((1.10, "tab:blue"), (V_CELL, "tab:green"), (1.40, "tab:red")):
    axes[0].plot(xs * pb.L_CHANNEL * 1e3, pb.reference(V)(xs), color=col, lw=2.5, label=f"reference, {V:.2f} V")
    axes[0].plot(xs * pb.L_CHANNEL * 1e3, pinn_at(V), "k:", lw=2)
axes[0].plot(xc * pb.L_CHANNEL * 1e3, y_f, "s", ms=3, color="tab:green", label="finite volumes, 1.29 V")
axes[0].plot([], [], "k:", lw=2, label="PINN")
axes[0].set_xlabel("x along the channel  [mm]"); axes[0].set_ylabel("steam fraction"); axes[0].legend(frameon=False, fontsize=8)
axes[1].plot(xs * pb.L_CHANNEL * 1e3, (V_CELL - pb.nernst(y_ref)) / pb.ASR / 1e4, lw=2.5, color="tab:green")
axes[1].set_xlabel("x along the channel  [mm]"); axes[1].set_ylabel("local current density at 1.29 V  [A/cm²]")
plt.tight_layout(); plt.savefig(cc.output_path("Ex10.2_report.png"), dpi=150); plt.show()

**What you should see.** All three agree: at 1.29 V the steam fraction falls
to 0.27, 69 % of the steam is used, and the cell makes 5.21 g of hydrogen an
hour for 178 W. Finite volumes are within 0.0042 of the reference and the
network within about 0.0005 - and the same network is within about 0.001 at
every other voltage from 1.10 to 1.40 V. The local current density falls
along the channel from about 1.73 to 1.15 A/cm²: the cell starves towards the
outlet. The solver and finite volumes take milliseconds and no training; the
network needs about 20 s of training on a CPU, once, for the whole range.

---

## 6 · The best cell voltage

A higher voltage makes more hydrogen, uses more power for each gram, and runs
the outlet short of steam. Somewhere there is a voltage at which an hour of
operation is worth most. Take hydrogen at 3.00 €/kg and electricity at
0.065 €/kWh - **prices chosen for the exercise**:

$$J(V) = p_{H_2}\,\dot m_{H_2}(V) - p_{el}\,P(V)$$

It is L10.2's *Why a Differentiable Model Changes the Problem*. The network is
a formula for $y(x, V)$, so $J$ is a formula in $V$, and **autograd gives
$dJ/dV$**. The cell below starts at 1.15 V and climbs that gradient. A
classical solver gives no gradient: to check the answer it is run 61 times,
at 61 voltages.

In [ ]:
xq = to_tensor(np.linspace(0, 1, 201).reshape(-1, 1))         # 201 points along the channel, for the integral
wq = torch.full_like(xq, 1 / 200); wq[0] = wq[-1] = 0.5 / 200 # the trapezoidal rule

def value(v):
    # the worth of an hour at scaled voltage v, in euro cents, as a function torch can differentiate
    V = pb.voltage(v)
    y = model(torch.cat([xq, v.expand(len(xq), 1)], 1))       # the network's steam fraction along the channel
    current = ((V - pb.nernst(y)) / pb.ASR * wq).sum() * pb.L_CHANNEL * pb.W_CELL   # the cell's current, A
    hydrogen = current / (2 * pb.FARADAY) * pb.M_H2 * 1000 * 3600                   # g/h, by Faraday's law
    return 100 * (pb.PRICE_H2 * hydrogen / 1000 - pb.PRICE_EL * V * current / 1000)

for p in model.parameters():
    p.requires_grad_(False)                                   # the network is finished: only the voltage moves now
v = torch.nn.Parameter(torch.full((1, 1), (1.15 - pb.V_LOW) / (pb.V_HIGH - pb.V_LOW), device=DEVICE))   # start at 1.15 V
opt = torch.optim.Adam([v], lr=0.02)                          # an optimiser for one number
t0 = time.perf_counter()
for step in range(200):
    opt.zero_grad()                                           # clear the old gradient
    (-value(v)).sum().backward()                              # dJ/dV by autograd, through the network
    opt.step()                                                # one step uphill
    v.data.clamp_(0.0, 1.0)                                   # stay inside the range the network was trained on
t_opt = time.perf_counter() - t0
V_best, J_best = float(pb.voltage(v).detach()), float(value(v).detach())   # the voltage it settled on, and its worth

t0 = time.perf_counter()
scan = np.linspace(pb.V_LOW, pb.V_HIGH, 61)                   # the check: the reference solver at 61 voltages
J_scan = []
for V in scan:
    n = pb.cell_numbers(xs, pb.reference(V)(xs), V); J_scan.append(pb.value_per_hour(n["hydrogen"], n["power"]))
t_scan = time.perf_counter() - t0
k = int(np.argmax(J_scan))
print(f"by the network's gradient : best voltage {V_best:.2f} V, worth {J_best:.2f} cent/h   ({t_opt:.1f} s, 200 steps)")
print(f"by 61 runs of the solver  : best voltage {scan[k]:.2f} V, worth {J_scan[k]:.2f} cent/h   ({t_scan:.1f} s)")
print(f"thermoneutral voltage     : {pb.V_TN:.2f} V")

**What you should see.** Both routes put the best voltage at about 1.30 V,
where an hour is worth about 0.41 cent for this one cell - next to the
thermoneutral voltage, 1.29 V. The curve is flat near its top: from 1.25 to
1.35 V the worth changes by a few per cent.

---

## 7 · What the notebook says

* **The current is an answer, not an input.** At 1.29 V the local current
  density falls from 1.73 A/cm² at the inlet to 1.15 at the outlet, because
  the Nernst potential rises as the steam is used. A model with one
  composition for the whole cell cannot show that.
* **All three methods agree**, to the agreed 0.005 and better: 69 % of the
  steam used, 5.21 g of hydrogen an hour, 178 W.
* **For one voltage, a solver is the tool.** The boundary-value solver gives
  the channel in a tenth of a second to twelve figures; finite volumes in a
  millisecond to half a per cent.
* **The network was trained once for every voltage.** That is what it has
  that a solver has not: the solver starts again at each voltage, and the
  network is one function of both.
* **And that function has a gradient.** The best voltage came from climbing
  $dJ/dV$ through the network. With one number to choose, 61 runs of the
  solver do as well; with an hourly schedule of 24 numbers, they do not.
* **The answer is only as good as two estimates.** The resistance and the
  diffusivity were chosen, not measured, and so were the prices. The method
  carries over to a real cell; these numbers do not.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex10.2_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L10.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The cell runs as an electrolyser at a fixed voltage. Why does the local current density fall along the channel, and by how much did it fall at 1.29 V? (-> L10.2 Q2)": """
""",
    "What is special about 1.29 V, and what would the cell do with heat at 1.10 V and at 1.40 V? (-> L10.2 Q3)": """
""",
    "Section 6 found one best voltage for fixed prices. What would make it a schedule and not a setting, and which cost is missing from the objective used here? (-> L10.2 Q4)": """
""",
    "How was the best voltage found without running a solver, how was it checked, and which of the notebook's numbers would you refuse to promise for a real cell? (-> L10.2 Q3)": """
""",
    "To conclude: for which task in this notebook would you use the boundary-value solver, and for which the network, and what changes when there are 24 voltages to choose and not one? (-> L10.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | steam at the outlet | used | hydrogen | power | worst error | one solution | training |", "|---|---|---|---|---|---|---|---|",
            f"| reference, boundary-value solver | {n_ref['outlet']:.2f} | {100 * n_ref['used']:.0f} % | {n_ref['hydrogen']:.2f} g/h | {n_ref['power']:.0f} W | - | {t_ref:.3f} s | - |",
            f"| finite volumes, 50 cells | {n_fvm['outlet']:.2f} | {100 * n_fvm['used']:.0f} % | {n_fvm['hydrogen']:.2f} g/h | {n_fvm['power']:.0f} W | {e_fvm:.4f} | {t_fvm:.3f} s | - |",
            f"| PINN, 4 x 32, any voltage from 1.10 to 1.40 V | {n_pin['outlet']:.2f} | {100 * n_pin['used']:.0f} % | {n_pin['hydrogen']:.2f} g/h | {n_pin['power']:.0f} W | {e_pin:.4f} | {t_pin:.3f} s | {train_time:.0f} s |",
            "", f"Best cell voltage: {V_best:.2f} V by the network's gradient ({J_best:.2f} cent/h), "
                f"{scan[k]:.2f} V by 61 runs of the reference solver ({J_scan[k]:.2f} cent/h)"]
    out = ["# Ex_10.2 — The Gas Channel of a Solid Oxide Cell", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex10.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex10.2_report.md into Ex10.2_report.pdf, with any figure
# saved as Ex10.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex10.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex10.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex10.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex10.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP10.2A · The channel with its temperature

**The problem.** The channel of this set was isothermal, and its cell was one resistance. A real cell is not: the reaction heats
or cools it depending on which side of the thermoneutral voltage it runs, and
the temperature changes the resistance in turn. Here the channel carries its
temperature as well as its composition, coupled both ways.

| | |
|---|---|
| cell | this set's cell model, 10 × 10 cm, electrolysis (SOEC), inlet at 800 °C, 10 % hydrogen, 90 % steam |
| flow | 8.6e-04 mol/s at the inlet - 60 % steam conversion at 1 A/cm² |
| voltage | held at 1.20, 1.29 (thermoneutral) and 1.35 V |
| heat | $q = i\,(V - V_{\text{tn}})$; the cell adiabatic, gas and cell at one temperature |

The adiabatic cell and the flow's heat capacity (5 × 35 J/mol·K per mole of
fuel-side flow) are assumptions for the project, and every value marked
`ESTIMATED` in `problem.py` stays so.

**Why a plain PINN is not enough.** The current is not an input but the answer to an equation at every point -
the cell voltage is fixed, the local Nernst potential and resistance are not -
and the temperature feeds back into it along the whole channel.

**What you build.** A PINN for the steam fraction and the temperature along the channel, with the
local current from the cell model at every point, coupled through the
reaction and its heat, at the three voltages.

**The ground truth you get.** `MP10.2A_truth.npz`, from `tools/miniprojects/ex102_truth.py`: the march
along the channel as an ODE (scipy's solve_ivp, tolerance 1e-10), the local
current solved at every point; the temperature and the current along $x$.

**Worked example.** 

| voltage | outlet temperature | current, inlet → outlet | total | steam converted |
|---|---|---|---|---|
| 1.20 V | 761.20 °C | 0.98 → 0.51 A/cm² | 68.6 A | 45.9 % |
| 1.29 V | 803.23 °C | 1.24 → 0.85 A/cm² | 100.7 A | 67.4 % |
| 1.35 V | 856.64 °C | 1.41 → 1.15 A/cm² | 131.5 A | 88.0 % |

Below the thermoneutral voltage the electrolyser cools itself by 39 K, its
resistance rises, and its current nearly halves along the channel; above it,
it heats by 57 K. Steps of $L$/25, $L$/50 and $L$/100 agree to 0.001 K.

**Requirements.**

* The outlet temperature within **1 K** and the temperature within **2 K** along the channel, at all three voltages.
* The total current within **1 %**.
* Say which voltage your network found hardest, and why.

**What you hand in.** The temperature and current profiles against the ground truth's at the three
voltages, and how the answer moves when you change one `ESTIMATED` parameter by
20 %.

### MP10.2B · The degradation law from long records

**The problem.** The lecture's degradation law - Arrhenius in temperature, a power law in
current - is in this set's `cell_model.py`, with parameters marked `ESTIMATED`. Here they are the unknowns.
Long voltage records from cells held at fixed temperature and current,
drifting as their resistance grows, are what a test lab would give.

| | |
|---|---|
| law | $d(\text{ASR})/dt = k_0\,e^{-E/RT}\,|i|^n$; $k_0$, $E$ and $n$ hidden |
| records | the cell voltage every 10 h for 3000 h, noise 0.5 mV |
| case 1 | 750 and 800 °C, 0.5 and 1.0 A/cm² - four cells |
| case 2 | 800 °C only, 0.5 and 1.0 A/cm² |
| case 3 | 800 °C, 1.0 A/cm² only |

**Why a plain PINN is not enough.** Temperature and current act together in the drift. A test at one temperature
cannot separate $E$ from $k_0$, and one at one current cannot give $n$ - and
the fit still returns numbers.

**What you build.** An inverse model: the full cell model (`cell_model.py`) with the three parameters
trainable - as $\log k_0$, $E$ and $n$ - fitted to the drifting records. You
report each with its uncertainty and say which the data does not determine.

**The ground truth you get.** `MP10.2B_readings.npz` to fit and `MP10.2B_truth.npz` with the hidden
values, from the same script: the course's own model run over 3000 h.

**Worked example.** Over 3000 h the voltage drifts 1.2 and 6.7 mV at 750 °C, 1.3 and 7.5 mV at
800 °C (0.5 and 1.0 A/cm²). In case 2, raising $E$ by 20 % and adjusting $k_0$
to match changes the records by exactly 0 V: from one temperature, $E$ cannot
be found.

**Requirements.**

* Case 1: $E$ and $n$ within **10 %**, and the drift at 775 °C and 0.75 A/cm² within **10 %**.
* Cases 2 and 3: show what cannot be recovered, and the smallest test plan that would recover it.

**What you hand in.** The recovered parameters with uncertainties in each case, the drift they
predict at a point you did not test, and the test plan you would ask for.